In [5]:
import time
import cv2
from movenet_trt import MoveNetTRT, KEYPOINT_NAMES, draw_keypoints
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit  # noqa
import numpy as np

ENGINE_FP16_PATH = "output/pose_fp16.engine"  # main output

# Use a real image if available, else a random image
TEST_IMG = "data/test_imgs/test1.jpg"
if os.path.exists(TEST_IMG):
    frame = cv2.imread(TEST_IMG)
    print("Using {}, shape={}".format(TEST_IMG, frame.shape))
else:
    frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
    print("No test.jpg, using random 480x640 image")

# Load engine
movenet_fp16 = MoveNetTRT(ENGINE_FP16_PATH, img_size=192)

Using data/test_imgs/test1.jpg, shape=(342, 608, 3)
[OK] MoveNetTRT loaded: output/pose_fp16.engine
     input : (1, 3, 192, 192)
     output: heatmap (1, 17, 48, 48)
     output: center (1, 1, 48, 48)
     output: regs (1, 34, 48, 48)
     output: offsets (1, 34, 48, 48)


In [6]:
def benchmark(model, frame, n_warmup=10, n_iter=100, label=""):
    # warmup (first calls are slow due to allocation/JIT)
    for _ in range(n_warmup):
        model.infer(frame)

    # timed runs
    times = []
    for _ in range(n_iter):
        t0 = time.time()
        model.infer(frame)
        times.append(time.time() - t0)
    times = np.array(times) * 1000   # to ms

    print("\n=== {} ===".format(label))
    print("  iters: {}".format(n_iter))
    print("  mean : {:.2f} ms  ({:5.1f} FPS)".format(times.mean(), 1000 / times.mean()))
    print("  p50  : {:.2f} ms".format(np.percentile(times, 50)))
    print("  p90  : {:.2f} ms".format(np.percentile(times, 90)))
    print("  p99  : {:.2f} ms".format(np.percentile(times, 99)))
    print("  min  : {:.2f} ms".format(times.min()))
    print("  max  : {:.2f} ms".format(times.max()))
    return times

fp16_times = benchmark(movenet_fp16, frame, label="TRT FP16")


=== TRT FP16 ===
  iters: 100
  mean : 30.21 ms  ( 33.1 FPS)
  p50  : 29.17 ms
  p90  : 34.50 ms
  p99  : 40.66 ms
  min  : 24.45 ms
  max  : 41.88 ms


In [ ]:
kpts = movenet_fp16.infer(frame)
print("output shape: {}, dtype: {}".format(kpts.shape, kpts.dtype))
print("\nkeypoints:")
for k, (x, y, s) in enumerate(kpts):
    print("  {:2d} {:14s}  ({:6.1f}, {:6.1f})  score={:.3f}".format(
        k, KEYPOINT_NAMES[k], x, y, s))

# If the input was a real image, save the visualization for visual inspection.
if os.path.exists(TEST_IMG):
    vis = draw_keypoints(frame, kpts)
    cv2.imwrite("trt_test_out.jpg", vis)
    print("\nVisualization saved: trt_test_out.jpg")

In [ ]:
# ============================================================
# SET TO None to skip this comparison.
# Otherwise point at your fire717 .pth file (the one you used
# to generate the ONNX).
# ============================================================
PTH_PATH = None    # e.g. "movenet/output/movenet.pth"

if PTH_PATH is None or not os.path.exists(PTH_PATH or ""):
    print("[SKIP] PyTorch baseline comparison")
    print("       Set PTH_PATH above to enable it.")
else:
    import torch
    # NOTE: this depends on fire717's model class. The cleanest way is to
    # import it from movenet/lib/. Adapt the import to your repo:
    #
    #   sys.path.insert(0, os.path.abspath('movenet'))
    #   from lib.movenet_mobilenetv2 import MoveNet
    #   model = MoveNet(num_classes=17, ft_size=48)
    #   model.load_state_dict(torch.load(PTH_PATH, map_location='cuda'))
    #   model = model.cuda().eval()
    #
    # Once `model` is loaded, re-implement preprocess/postprocess (or
    # reuse fire717's predict.py logic) and benchmark with the same
    # `benchmark()` helper above.
    print("PyTorch baseline scaffolding — fill in the model loading for your repo.")
    print("Expected result on Nano: ~5-10x slower than TRT FP16.")

In [ ]:
BUILD_FP32 = False    # flip to True to actually build it (takes another 5-15 min)

if BUILD_FP32:
    build_engine(
        onnx_path    = ONNX_PATH,
        engine_path  = ENGINE_FP32_PATH,
        fp16         = False,
        workspace_gb = 1,
        rebuild      = False,
    )
    movenet_fp32 = MoveNetTRT(ENGINE_FP32_PATH, img_size=192)
    fp32_times = benchmark(movenet_fp32, frame, label="TRT FP32")

    speedup = fp32_times.mean() / fp16_times.mean()
    print("\nFP16 vs FP32 speedup: {:.2f}x".format(speedup))

    # Compare keypoint outputs
    k_fp16 = movenet_fp16.infer(frame)
    k_fp32 = movenet_fp32.infer(frame)
    diff = np.abs(k_fp16[:, :2] - k_fp32[:, :2])
    print("\nFP16 vs FP32 keypoint position diff (pixels):")
    print("  mean: {:.3f}, max: {:.3f}".format(diff.mean(), diff.max()))
    print("  (sub-pixel diff is normal; should be < 1 pixel typically)")
else:
    print("[SKIP] FP32 baseline. Set BUILD_FP32=True to enable.")